# Preparación, limpieza y estructuración de los datos de propiedad horizontal en Medellín

**Evidencia:** GA1-220501094-01-AA1-EV07 – Preparación, limpieza y estructuración de datos del proyecto individual con Pandas

**Aprendiz:** Maicol Esneider Posada Callejas · **Ficha:** 3223873 · **Instructor:** Carlos Navia

**Pregunta analítica del proyecto:** ¿En qué comunas de Medellín se concentran los predios habitacionales en propiedad horizontal y qué proporción representan frente al total de predios habitacionales de cada comuna?

**Pregunta orientadora de la actividad:** ¿Cómo preparar y estructurar los datos del proyecto individual para obtener información confiable que permita responder la pregunta analítica planteada?

Este cuaderno aplica las técnicas de las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python* a los cuatro datasets del proyecto. Parte de los problemas encontrados en el diagnóstico de calidad (`data_quality.ipynb` y `reports/data_quality.md`).

**Contenido**

0. Carga de los datos originales y estado inicial
1. Limpieza y estandarización (Lección 2)
2. Filtrado, ordenamiento, agrupación e indicadores (Lección 3)
3. Combinación y estructuración (Lección 4)
4. Comparación de la calidad antes y después

## 0. Carga de los datos originales y estado inicial

Los cuatro archivos están en `data/raw/` y no se modifican. Primero se cargan tal como vienen, para dejar registrado su estado inicial.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUTA = Path('../data/raw')
SALIDA = Path('../data/processed')
SALIDA.mkdir(exist_ok=True)

predios_crudo     = pd.read_csv(RUTA / 'informacion_predios.csv')
lotes_crudo       = pd.read_csv(RUTA / 'informacion_lotes.csv')
propiedades_crudo = pd.read_csv(RUTA / 'medellin_properties.csv')
inmuebles_crudo   = pd.read_csv(RUTA / 'houses_medellin.csv')

print('pandas', pd.__version__)

pandas 3.0.5


In [2]:
# Textos que el diagnóstico encontró como valores de relleno (datos faltantes disfrazados)
RELLENOS = ['--', 'INVALID', 'No definida', 'Otro', 'Más de 10']

def perfil(df, nombre):
    """Resume el estado de calidad de un DataFrame en una fila."""
    return {
        'dataset': nombre,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'filas_duplicadas': int(df.duplicated().sum()),
        'valores_relleno': int(df.isin(RELLENOS).sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    }

estado_inicial = pd.DataFrame([
    perfil(predios_crudo, 'predios'),
    perfil(lotes_crudo, 'lotes'),
    perfil(propiedades_crudo, 'propiedades_kaggle'),
    perfil(inmuebles_crudo, 'inmuebles_kaggle'),
])
estado_inicial

,dataset,filas,columnas,nulos,filas_duplicadas,valores_relleno,memoria_mb
0,predios,1041413,15,5800,0,590022,156.9
1,lotes,308129,8,0,0,0,22.8
2,propiedades_kaggle,9999,12,6028,100,127,1.4
3,inmuebles_kaggle,430,14,2365,0,128,0.1


Problemas del diagnóstico de calidad que se tratan en este cuaderno:

| Dataset | Problemas |
|---|---|
| Predios | `CBML` y `Matricula` leídos como números (se pierde el cero inicial), `NroPropietarios` siempre en 0, "--" en `Urbanizacion`, estrato 0, comunas mal escritas, códigos de `UsoPredial` sin descripción |
| Lotes | `CBML` como número, `ZonaGeo` como decimal, lotes con área 0 y con pisos o área construida incoherentes |
| Propiedades (Kaggle) | Duplicados, coordenadas como texto con "INVALID" y fuera de Medellín, "Más de 10" en garajes, áreas y precios imposibles, estrato 0 |
| Inmuebles (Kaggle) | Columna de índice sobrante, montos y áreas como texto, filas que solo tienen el precio, "No definida" y "Otro" como valores de relleno |

---
# 1. Limpieza y estandarización (Lección 2)

Para cada dataset se construye un pipeline: una cadena de funciones pequeñas unidas con `pipe()`. Cada función hace un solo paso de limpieza y la función `anotar` registra el estado de la tabla después de cada paso. Al final, las reglas de calidad se comprueban con `assert`.

## 1.1 Registro de cambios y funciones reutilizables

In [3]:
registro = []

def anotar(df, tabla, paso):
    """Guarda filas, nulos, duplicados y memoria de la tabla en el registro de cambios."""
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'columnas': df.shape[1],
        'nulos': int(df.isna().sum().sum()),
        'duplicados': int(df.duplicated().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df


def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)


def renombrar(df, nombres):
    return df.rename(columns=nombres)


def estandarizar_texto(df, columnas, correcciones=None):
    """Quita espacios sobrantes, corrige nombres mal escritos y deja Mayúscula Inicial."""
    df = df.copy()
    for col in columnas:
        texto = (df[col].astype('string')
                 .str.replace(r'\s+', ' ', regex=True)
                 .str.strip()
                 .str.upper())
        if correcciones:
            texto = texto.replace(correcciones)
        df[col] = texto.str.title()
    return df


def codigo_cbml(serie):
    """Convierte el CBML leído como número a texto de 11 dígitos, recuperando el cero inicial."""
    return serie.astype('int64').astype(str).str.zfill(11)


CORRECCIONES_COMUNAS = {'MANRRIQUE': 'MANRIQUE', 'LA AMERIA': 'LA AMÉRICA'}

`estandarizar_texto`, `quitar_duplicados`, `renombrar` y `codigo_cbml` se usan en varios datasets. Las columnas se renombran a un estándar único en minúsculas con guion bajo, porque el diagnóstico mostró que los nombres del CSV no coinciden con los del diccionario de MEData.

## 1.2 Información Predios

**Decisiones de limpieza:**

| Problema | Decisión | Justificación |
|---|---|---|
| `CBML` como número | Convertir a texto de 11 dígitos | Es un código: comuna (2) + barrio (2) + manzana (3) + lote (4) |
| `Matricula` como número | Convertir a texto | Es un identificador, no una cantidad |
| `NroPropietarios` siempre 0 | Eliminar la columna | No aporta información |
| "--" en `Urbanizacion` | Convertir a nulo | Es un valor de relleno |
| Estrato 0 | Convertir a nulo | El estrato solo se asigna a vivienda; 0 significa "sin estrato" |
| "MANRRIQUE", "LA AMERIA" | Corregir | Son errores de escritura |
| Valores nulos en `Urbanizacion` | Conservar | Solo es un nombre descriptivo; no se usa en el análisis |

In [4]:
NOMBRES_PREDIOS = {
    'codComuna': 'cod_comuna', 'codBarrio': 'cod_barrio', 'Manzana': 'manzana', 'CedCat': 'lote',
    'CBML': 'cbml', 'Matricula': 'matricula', 'PorDes': 'por_des', 'Estrato': 'estrato',
    'Urbanizacion': 'urbanizacion', 'UsoPredial': 'uso_predial', 'IndicadorU/R': 'zona',
    'NroPropietarios': 'nro_propietarios', 'nomComuna': 'comuna', 'DS_BARRIO': 'barrio',
    'AvaluoTotal': 'avaluo_total',
}

def tipos_predios(df):
    df = df.copy()
    df['cbml'] = codigo_cbml(df['cbml'])
    df['matricula'] = df['matricula'].astype(str)
    return df.astype({'cod_comuna': 'int8', 'cod_barrio': 'int8', 'manzana': 'int16', 'lote': 'int16',
                      'uso_predial': 'int8', 'zona': 'category'})

def quitar_columna_vacia(df):
    assert (df['nro_propietarios'] == 0).all()
    return df.drop(columns='nro_propietarios')

def rellenos_a_nulo_predios(df):
    df = df.copy()
    df['urbanizacion'] = df['urbanizacion'].replace('--', pd.NA)
    df['estrato'] = df['estrato'].replace(0, pd.NA).astype('Int8')
    return df

def clasificar_territorio(df):
    df = df.copy()
    df['tipo_territorio'] = np.where(df['cod_comuna'] <= 16, 'comuna', 'corregimiento')
    return df

predios = (predios_crudo
    .pipe(anotar, 'predios', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'predios', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_PREDIOS)
    .pipe(tipos_predios)
    .pipe(anotar, 'predios', '2 nombres y tipos')
    .pipe(quitar_columna_vacia)
    .pipe(rellenos_a_nulo_predios)
    .pipe(anotar, 'predios', '3 rellenos a nulo')
    .pipe(estandarizar_texto, ['comuna', 'barrio', 'urbanizacion'], CORRECCIONES_COMUNAS)
    .pipe(clasificar_territorio)
    .pipe(anotar, 'predios', '4 texto estandarizado'))

predios.head()

,cod_comuna,cod_barrio,manzana,lote,cbml,matricula,por_des,estrato,urbanizacion,uso_predial,zona,comuna,barrio,avaluo_total,tipo_territorio
0,11,5,55,1,11050550001,257792,0.085,<NA>,Torres De Rio No 1 Y 2,10,U,Laureles,Los Conquistadores,17500000.0,comuna
1,11,5,55,1,11050550001,257821,0.085,<NA>,Torres De Rio No 1 Y 2,10,U,Laureles,Los Conquistadores,17500000.0,comuna
2,11,9,28,27,11090280027,258113,33.823,5,<NA>,1,U,Laureles,Las Acacias,326925000.0,comuna
3,80,98,11,15,80980110015,1245992,0.020,<NA>,Reservas Del Prado Etapa 2,10,U,San Antonio De Prado,Zona Expansion San Antonio,6342000.0,corregimiento
4,80,98,11,15,80980110015,1246000,0.020,<NA>,Reservas Del Prado Etapa 2,10,U,San Antonio De Prado,Zona Expansion San Antonio,6342000.0,corregimiento


In [5]:
print(predios.dtypes)
print('\nComunas y corregimientos:', sorted(predios['comuna'].unique().tolist()))

cod_comuna             int8
cod_barrio             int8
manzana               int16
lote                  int16
cbml                    str
matricula               str
por_des             float64
estrato                Int8
urbanizacion         string
uso_predial            int8
zona               category
comuna               string
barrio               string
avaluo_total        float64
tipo_territorio         str
dtype: object

Comunas y corregimientos: ['Altavista', 'Aranjuez', 'Belen', 'Buenos Aires', 'Castilla', 'Doce De Octubre', 'El Poblado', 'Guayabal', 'La América', 'La Candelaria', 'Laureles', 'Manrique', 'Palmitas', 'Popular', 'Robledo', 'San Antonio De Prado', 'San Cristobal', 'San Javier', 'Santa Cruz', 'Santa Elena', 'Villa Hermosa']


### Uso predial: ¿qué código corresponde a vivienda?

El diagnóstico encontró que `UsoPredial` viene en códigos sin tabla de equivalencias. Como el estrato socioeconómico en Colombia solo se asigna a la vivienda, se cruza el código de uso con la existencia de estrato:

In [6]:
uso_vs_estrato = (predios
    .assign(tiene_estrato=predios['estrato'].notna())
    .groupby('uso_predial')
    .agg(predios=('matricula', 'size'),
         con_estrato=('tiene_estrato', 'sum'),
         por_des_mediana=('por_des', 'median')))
uso_vs_estrato['pct_con_estrato'] = (uso_vs_estrato['con_estrato'] / uso_vs_estrato['predios'] * 100).round(2)
uso_vs_estrato.sort_values('predios', ascending=False)

,predios,con_estrato,por_des_mediana,pct_con_estrato
uso_predial,,,,
1,645912,645907,24.4765,100.00
10,240381,2158,0.0700,0.90
2,99326,1386,0.4750,1.40
4,21022,249,100.0000,1.18
9,16788,94,100.0000,0.56
3,6989,581,100.0000,8.31
5,6983,3,100.0000,0.04
8,3702,2,100.0000,0.05
7,307,9,100.0000,2.93


**Resultado:** el código **1** tiene 645.912 predios y **645.907 tienen estrato (100 %)**. En los demás códigos, como mucho el 8,3 % de los predios tiene estrato. Además, los códigos 10 y 2 tienen porcentajes de copropiedad muy pequeños (mediana de 0,07 y 0,48), típicos de parqueaderos, cuartos útiles y locales dentro de edificios.

**Decisión:** se toma `uso_predial = 1` como **vivienda**. Es una inferencia sustentada en los datos, no una tabla oficial; si Catastro entrega la tabla de códigos, este criterio se debe confirmar.

In [7]:
predios['es_vivienda'] = predios['uso_predial'] == 1
print('Predios de vivienda:', predios['es_vivienda'].sum())

Predios de vivienda: 645912


## 1.3 Información Lotes

| Problema | Decisión | Justificación |
|---|---|---|
| `CBML` como número | Convertir a texto de 11 dígitos | Es la clave para unir con Predios |
| `ZonaGeo` como decimal | Convertir a texto sin el ".0" | Es un código |
| Área del lote igual a 0 | Convertir a nulo | Un lote no puede medir 0 m² |
| Área construida 0 con pisos, o área construida con 0 pisos | Marcar con una columna `pisos_incoherentes` | No se sabe cuál de los dos datos está mal; se conservan y se excluyen de los cálculos de pisos |
| Nombres de comuna mal escritos | Corregir | Igual que en Predios |

In [8]:
NOMBRES_LOTES = {'CBML': 'cbml', 'codComuna': 'cod_comuna', 'nomComuna': 'comuna', 'AreaLote': 'area_lote',
                 'AreaCons': 'area_construida', 'Zona': 'zona_codigo', 'ZonaGeo': 'zona_geo',
                 'numPisos': 'num_pisos'}

def tipos_lotes(df):
    df = df.copy()
    df['cbml'] = codigo_cbml(df['cbml'])
    df['zona_geo'] = df['zona_geo'].astype('Int64').astype('string')
    return df.astype({'cod_comuna': 'int8', 'zona_codigo': 'int16', 'num_pisos': 'int16'})

def marcar_inconsistencias_lotes(df):
    df = df.copy()
    df['area_lote'] = df['area_lote'].replace(0, np.nan)
    df['pisos_incoherentes'] = (((df['area_construida'] == 0) & (df['num_pisos'] > 0)) |
                                ((df['area_construida'] > 0) & (df['num_pisos'] == 0)))
    return df

lotes = (lotes_crudo
    .pipe(anotar, 'lotes', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'lotes', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_LOTES)
    .pipe(tipos_lotes)
    .pipe(anotar, 'lotes', '2 nombres y tipos')
    .pipe(marcar_inconsistencias_lotes)
    .pipe(anotar, 'lotes', '3 inconsistencias marcadas')
    .pipe(estandarizar_texto, ['comuna'], CORRECCIONES_COMUNAS)
    .pipe(anotar, 'lotes', '4 texto estandarizado'))

print('Lotes con área 0 convertida a nulo:', lotes['area_lote'].isna().sum())
print('Lotes con pisos incoherentes:', lotes['pisos_incoherentes'].sum())
print('Longitud de zona_geo:')
print(lotes['zona_geo'].str.len().value_counts())
lotes.head()

Lotes con área 0 convertida a nulo: 80
Lotes con pisos incoherentes: 1111
Longitud de zona_geo:
zona_geo
15    306559
3       1566
8          2
7          1
13         1
Name: count, dtype: Int64


,cbml,cod_comuna,comuna,area_lote,area_construida,zona_codigo,zona_geo,num_pisos,pisos_incoherentes
0,01010020003,1,Popular,33.0,85.0,440,122121422212440,2,False
1,01010020010,1,Popular,145.0,188.0,440,122121422212440,2,False
2,01010020014,1,Popular,55.0,173.0,440,203121422313440,3,False
3,01010030013,1,Popular,123.0,102.0,440,111131422212440,2,False
4,01010030021,1,Popular,51.0,129.0,440,111131422212440,2,False


`zona_geo` quedó como texto: 306.559 lotes tienen el código completo de 15 dígitos y 1.566 tienen un código de 3 dígitos (igual al formato de `zona_codigo`). Esa diferencia se conserva y se documenta, porque `zona_geo` no se usa para responder la pregunta.

## 1.4 Medellín Properties 2023 (Kaggle)

| Problema | Decisión |
|---|---|
| 100 filas duplicadas, más las que aparecen al estandarizar | Eliminar antes y después de limpiar |
| Coordenadas como texto con "INVALID" | Convertir a número; "INVALID" queda como nulo |
| Coordenadas fuera de Medellín | Convertir a nulo (el inmueble se conserva) |
| "Más de 10" en garajes | Convertir a nulo y marcar en `garajes_mas_de_10` |
| Área 0, estrato 0 y barrio "Otro" | Convertir a nulo |
| `age` es un código de rango | Convertir a categoría ordenada |
| Precios y áreas imposibles | Marcar en `valor_atipico`; se excluyen de los cálculos |

In [9]:
NOMBRES_PROPIEDADES = {'neighbourhood': 'barrio', 'latitude': 'latitud', 'longitude': 'longitud',
                       'property_type': 'tipo', 'price': 'precio', 'rooms': 'habitaciones', 'baths': 'banos',
                       'area': 'area', 'administration_price': 'administracion', 'age': 'antiguedad_codigo',
                       'garages': 'garajes', 'stratum': 'estrato'}

def coordenadas(df):
    df = df.copy()
    df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
    df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
    dentro = df['latitud'].between(6.1, 6.4) & df['longitud'].between(-75.75, -75.45)
    df.loc[~dentro, ['latitud', 'longitud']] = np.nan
    return df

def corregir_valores_propiedades(df):
    df = df.copy()
    df['garajes_mas_de_10'] = df['garajes'] == 'Más de 10'
    df['garajes'] = pd.to_numeric(df['garajes'], errors='coerce').astype('Int8')
    df['area'] = df['area'].replace(0, np.nan)
    df['estrato'] = df['estrato'].replace(0, pd.NA).astype('Int8')
    df['habitaciones'] = df['habitaciones'].astype('Int8')
    df['banos'] = df['banos'].astype('Int8')
    df['antiguedad_codigo'] = pd.Categorical(df['antiguedad_codigo'].astype('Int8'),
                                             categories=[0, 1, 2, 3, 4], ordered=True)
    df['valor_atipico'] = (~df['precio'].between(5e7, 1e11)) | (df['area'] > 2000)
    return df

def barrio_desconocido(df):
    df = df.copy()
    df['barrio'] = df['barrio'].replace('Otro', pd.NA)
    return df

propiedades = (propiedades_crudo
    .pipe(anotar, 'propiedades_kaggle', '0 original')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'propiedades_kaggle', '1 sin duplicados')
    .pipe(renombrar, NOMBRES_PROPIEDADES)
    .pipe(coordenadas)
    .pipe(anotar, 'propiedades_kaggle', '2 coordenadas')
    .pipe(corregir_valores_propiedades)
    .pipe(estandarizar_texto, ['barrio', 'tipo'])
    .pipe(barrio_desconocido)
    .pipe(anotar, 'propiedades_kaggle', '3 valores y texto')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'propiedades_kaggle', '4 duplicados que aparecieron al limpiar'))

print('Coordenadas válidas:', propiedades['latitud'].notna().sum(), 'de', len(propiedades))
print('Inmuebles con valor atípico:', propiedades['valor_atipico'].sum())
print(propiedades.dtypes)

Coordenadas válidas: 9727 de 9883
Inmuebles con valor atípico: 35
barrio                 string
latitud               float64
longitud              float64
tipo                   string
precio                  int64
habitaciones             Int8
banos                    Int8
area                  float64
administracion        float64
antiguedad_codigo    category
garajes                  Int8
estrato                  Int8
garajes_mas_de_10        bool
valor_atipico            bool
dtype: object


## 1.5 Precios inmuebles Medellín (Kaggle)

| Problema | Decisión |
|---|---|
| Columna `Unnamed: 0` (índice guardado por error) | Eliminar |
| Filas que solo tienen el precio | Eliminar: no aportan nada para relacionar la administración con el estrato o el área |
| Montos con `$`, `COP`, `*m²`, puntos de miles y comas decimales | Convertir a número |
| Áreas como texto con espacios y comas | Convertir a número; área privada 0 a nulo |
| "No definida" y "Otro" | Convertir a nulo |
| Antigüedad en rangos de texto | Convertir a categoría ordenada |
| Duplicados que aparecen después de limpiar | Eliminar |

In [10]:
NOMBRES_INMUEBLES = {'Habitaciones': 'habitaciones', 'Baños': 'banos', 'Estrato': 'estrato',
                     'Antigüedad': 'antiguedad', 'Piso N°': 'piso', 'Administración': 'administracion',
                     'Precio m²': 'precio_m2', 'Parqueaderos': 'parqueaderos', 'Estado': 'estado',
                     'Tipo de apartamento': 'tipo_apartamento', 'Precio': 'precio',
                     'Área construída (m²)': 'area_construida', 'Área privada (m²)': 'area_privada'}

def texto_a_numero(serie):
    """'$ 4.032.258,06*m²' -> 4032258.06 ; 'No definida' -> nulo"""
    limpio = (serie.astype('string')
              .str.replace(r'[^0-9,]', '', regex=True)
              .str.replace(',', '.', regex=False)
              .replace('', pd.NA))
    return pd.to_numeric(limpio, errors='coerce')

def quitar_indice_y_filas_vacias(df):
    df = df.drop(columns='Unnamed: 0')
    sin_precio = df.drop(columns='Precio')
    return df[sin_precio.notna().any(axis=1)].reset_index(drop=True)

def convertir_inmuebles(df):
    df = df.copy()
    for col in ['administracion', 'precio_m2', 'area_construida', 'area_privada']:
        df[col] = texto_a_numero(df[col])
    df['area_privada'] = df['area_privada'].replace(0, np.nan)
    df['piso'] = pd.to_numeric(df['piso'], errors='coerce').astype('Int8')
    for col in ['habitaciones', 'banos', 'estrato', 'parqueaderos']:
        df[col] = df[col].astype('Int8')
    df['antiguedad'] = pd.Categorical(df['antiguedad'],
        categories=['menor a 1 año', '1 a 8 años', '9 a 15 años', '16 a 30 años', 'más de 30 años'],
        ordered=True)
    df['valor_atipico'] = ~df['precio'].between(5e7, 1e11)
    return df

inmuebles = (inmuebles_crudo
    .pipe(anotar, 'inmuebles_kaggle', '0 original')
    .pipe(quitar_indice_y_filas_vacias)
    .pipe(anotar, 'inmuebles_kaggle', '1 sin índice ni filas vacías')
    .pipe(renombrar, NOMBRES_INMUEBLES)
    .pipe(convertir_inmuebles)
    .pipe(anotar, 'inmuebles_kaggle', '2 montos y áreas a número')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'inmuebles_kaggle', '3 sin duplicados'))

print(inmuebles.dtypes)
inmuebles.head()

habitaciones            Int8
banos                   Int8
estrato                 Int8
antiguedad          category
piso                    Int8
administracion         Int64
precio_m2            Float64
parqueaderos            Int8
estado                   str
tipo_apartamento         str
precio                 int64
area_construida      Float64
area_privada         Float64
valor_atipico           bool
dtype: object


,habitaciones,banos,estrato,antiguedad,piso,administracion,precio_m2,parqueaderos,estado,tipo_apartamento,precio,area_construida,area_privada,valor_atipico
0,1,2,4,1 a 8 años,3,<NA>,4032258.06,<NA>,NaN,NaN,250000000,62.0,62.0,False
1,1,2,6,9 a 15 años,<NA>,<NA>,6428571.43,2,NaN,NaN,675000000,105.0,<NA>,False
2,1,1,5,NaN,3,167000,4700000.0,2,NaN,NaN,235000000,50.0,<NA>,False
3,1,1,4,NaN,<NA>,<NA>,6187500.0,1,Excelente,NaN,198000000,32.0,<NA>,False
4,2,2,6,menor a 1 año,<NA>,180000,7421875.0,1,Excelente,NaN,475000000,64.0,64.0,False


## 1.6 Registro de cambios

In [11]:
registro_cambios = pd.DataFrame(registro)
registro_cambios.to_csv(SALIDA / 'cleaning_log.csv', index=False)
registro_cambios

,tabla,paso,filas,columnas,nulos,duplicados,memoria_mb
0,predios,0 original,1041413,15,5800,0,156.9
1,predios,1 sin duplicados,1041413,15,5800,0,156.9
2,predios,2 nombres y tipos,1041413,15,5800,0,132.5
3,predios,3 rellenos a nulo,1041413,14,986846,0,116.7
4,predios,4 texto estandarizado,1041413,15,986846,0,132.3
5,lotes,0 original,308129,8,0,0,22.8
6,lotes,1 sin duplicados,308129,8,0,0,22.8
7,lotes,2 nombres y tipos,308129,8,0,0,24.9
8,lotes,3 inconsistencias marcadas,308129,9,80,0,25.2
9,lotes,4 texto estandarizado,308129,9,80,0,25.3


**Interpretación del registro de cambios:**

- **Predios:** no se perdió ninguna fila. Los nulos suben de 5.800 a 986.846 porque los valores de relleno quedaron como nulos reales: 590.022 "--" en `urbanizacion` y 391.024 estratos en 0. Antes esos datos faltantes estaban escondidos; ahora pandas los reconoce. La memoria baja de 156,9 a 133,3 MB con tipos más pequeños.
- **Lotes:** no se perdió ninguna fila. Aparecen 80 nulos: los lotes con área 0.
- **Medellín Properties:** se eliminaron 100 duplicados exactos y otros 16 que aparecieron al limpiar coordenadas y texto (de 9.999 a 9.883 filas).
- **Precios inmuebles:** se eliminaron 121 filas que solo tenían el precio y 98 duplicados (de 430 a 211 filas). El dataset queda pequeño, pero confiable.

El registro se guardó en `data/processed/cleaning_log.csv`.

## 1.7 Validación con `assert`

In [12]:
def validar_predios(df):
    assert df['matricula'].is_unique, 'matrícula repetida'
    assert (df['cbml'].str.len() == 11).all(), 'CBML sin 11 dígitos'
    assert df['cbml'].str.isdigit().all(), 'CBML con caracteres no numéricos'
    assert df['estrato'].dropna().between(1, 6).all(), 'estrato fuera de 1 a 6'
    assert df['por_des'].between(0, 100).all(), 'porcentaje de copropiedad fuera de 0 a 100'
    assert not df['comuna'].isin(['Manrrique', 'La Ameria']).any(), 'comunas mal escritas'
    assert (df['urbanizacion'] != '--').all(), 'quedan valores de relleno'
    return 'Predios: validación superada'

def validar_lotes(df):
    assert df['cbml'].is_unique, 'CBML repetido'
    assert (df['cbml'].str.len() == 11).all(), 'CBML sin 11 dígitos'
    assert (df['area_lote'].dropna() > 0).all(), 'áreas en 0'
    return 'Lotes: validación superada'

def validar_propiedades(df):
    assert df.duplicated().sum() == 0, 'duplicados'
    assert df['latitud'].dropna().between(6.1, 6.4).all(), 'latitud fuera de Medellín'
    assert df['estrato'].dropna().between(1, 6).all(), 'estrato fuera de 1 a 6'
    assert pd.api.types.is_numeric_dtype(df['garajes']), 'garajes no numérico'
    return 'Propiedades: validación superada'

def validar_inmuebles(df):
    assert 'Unnamed: 0' not in df.columns, 'columna de índice'
    assert df.duplicated().sum() == 0, 'duplicados'
    for col in ['administracion', 'precio_m2', 'area_construida']:
        assert pd.api.types.is_numeric_dtype(df[col]), f'{col} no numérico'
    return 'Inmuebles: validación superada'

print(validar_predios(predios))
print(validar_lotes(lotes))
print(validar_propiedades(propiedades))
print(validar_inmuebles(inmuebles))

Predios: validación superada


Lotes: validación superada
Propiedades: validación superada
Inmuebles: validación superada


Los cuatro datasets cumplen todas las reglas de calidad definidas. Si alguna fallara, `assert` detendría el cuaderno con el mensaje del problema.